In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (keep original unix seconds as well)
for df in (df_metric, df_trace, df_log, df_error):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        df['ts'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True)

# Helper to build file-level meta summary: earliest/latest raw unix seconds and distinct cmdb_id list (max 50)
def file_meta(df):
    if df.shape[0] == 0:
        return pd.DataFrame([{
            'earliest_ts': None,
            'latest_ts': None,
            'distinct_cmdb_ids': []
        }])
    earliest = int(df['timestamp'].min())
    latest = int(df['timestamp'].max())
    cmdbs = list(pd.unique(df['cmdb_id']))[:50]
    return pd.DataFrame([{
        'earliest_ts': earliest,
        'latest_ts': latest,
        'distinct_cmdb_ids': cmdbs
    }])

# 1) File-level metas
metric_file_meta = file_meta(df_metric)
trace_file_meta = file_meta(df_trace)
log_file_meta = file_meta(df_log)
error_file_meta = file_meta(df_error)

# 2) metric.csv per (cmdb_id, kpi_name) stats including global P95 per group
if not df_metric.empty:
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    metric_agg = grp.agg(count='count', min_val='min', median_val='median', max_val='max').reset_index()
    # P95 computed on entire series per group (quantile)
    p95_series = grp.quantile(0.95).reset_index().rename(columns={'value': 'p95'})
    metric_agg = metric_agg.merge(p95_series, on=['cmdb_id', 'kpi_name'])
    metric_top20 = metric_agg.sort_values('count', ascending=False).head(20)
else:
    metric_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','count','min_val','median_val','max_val','p95'])

# 3) trace.csv per (cmdb_id, trace_name)
if not df_trace.empty:
    df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
    grp = df_trace.groupby(['cmdb_id', 'trace_name'])['value']
    trace_agg = grp.agg(count='count', min_val='min', median_val='median', max_val='max').reset_index()
    p95_series = grp.quantile(0.95).reset_index().rename(columns={'value': 'p95'})
    trace_agg = trace_agg.merge(p95_series, on=['cmdb_id', 'trace_name'])
    trace_top20 = trace_agg.sort_values('count', ascending=False).head(20)
else:
    trace_top20 = pd.DataFrame(columns=['cmdb_id','trace_name','count','min_val','median_val','max_val','p95'])

# 4) log.csv per (cmdb_id, log_name)
if not df_log.empty:
    df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
    grp = df_log.groupby(['cmdb_id', 'log_name'])['value']
    log_agg = grp.agg(count='count', min_val='min', median_val='median', max_val='max').reset_index()
    p95_series = grp.quantile(0.95).reset_index().rename(columns={'value': 'p95'})
    log_agg = log_agg.merge(p95_series, on=['cmdb_id', 'log_name'])
    log_top20 = log_agg.sort_values('count', ascending=False).head(20)
else:
    log_top20 = pd.DataFrame(columns=['cmdb_id','log_name','count','min_val','median_val','max_val','p95'])

# 5) error_logs.csv: up to 20 most frequent (cmdb_id, message) pairs with counts and earliest/latest timestamps
if not df_error.empty:
    grp = df_error.groupby(['cmdb_id', 'message'])
    err_agg = grp.agg(count=('timestamp','size'), earliest=('timestamp','min'), latest=('timestamp','max')).reset_index()
    error_top20 = err_agg.sort_values('count', ascending=False).head(20)
else:
    error_top20 = pd.DataFrame(columns=['cmdb_id','message','count','earliest','latest'])

# Keep outputs compact as required
metric_file_meta, metric_top20, trace_file_meta, trace_top20, log_file_meta, log_top20, error_file_meta, error_top20

```
Out[1]:
```
summary = (
    "Telemetry summary (plain English):\n\n"
    "Files and time range:\n"
    "- metric.csv: earliest=1705675920, latest=1705677360. Multiple components present (e.g., carts, carts-db, "
    "catalogue, catalogue-db, front-end, orders, payment, queue-master, user, shipping, session-db).\n"
    "- trace.csv: no trace records available.\n"
    "- log.csv: earliest=1705675920, latest=1705677360. Logs from components like catalogue, front-end, orders, "
    "payment, queue-master, user, shipping, carts, carts-db, session-db are present.\n"
    "- error_logs.csv: no error log records available.\n\n"
    "Key observations from metric.csv:\n"
    "- carts shows very high CPU values (p95 ≈ 7.934422e+01, max ≈ 7.958958e+01).\n"
    "- carts also shows very large disk I/O (p95 ≈ 5.924902e+04, max ≈ 1.163966e+05).\n"
    "- carts-db shows extremely large disk I/O (p95 ≈ 2.270569e+06, max ≈ 2.296373e+06).\n"
    "- carts memory values are large (median ≈ 2.198430e+08, p95 ≈ 2.219550e+08), indicating high memory usage magnitude.\n\n"
    "Key observations from log.csv:\n"
    "- log.error_count is zero for recorded components (no normalized log error counts detected).\n"
    "- Traffic/row counts are high at front-end (median 1966, p95 ≈ 2040) and present for carts (median 50, p95 ≈ 146.8), "
    "orders, payment, etc.\n\n"
    "Implication / recommended next steps:\n"
    "- The strongest signals point to resource pressure on the carts service and its database (carts-db), "
    "notably high CPU and very high disk I/O. These are the primary hotspots to investigate.\n"
    "- There are no recorded trace data or error log entries to explain application-level failures; collect traces and "
    "error logs if deeper request-level diagnosis is needed.\n"
    "- Investigate system-level metrics on carts and carts-db (CPU, I/O wait, disk throughput/latency, filesystem usage), "
    "and verify whether high traffic from front-end is driving the load.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   earliest_ts   latest_ts                                  distinct_cmdb_ids
0   1705675920  1705677360  [carts, carts-db, catalogue, catalogue-db, fro...,          cmdb_id    kpi_name  count       min_val    median_val       max_val           p95
0          carts         cpu     25  1.170727e+00  2.583651e+00  7.958958e+01  7.934422e+01
1          carts      diskio     25  0.000000e+00  0.000000e+00  1.163966e+05  5.924902e+04
3          carts  latency-50     25  1.266706e-02  1.523511e-02  2.043347e-02  1.948694e-02
4          carts  latency-90     25  2.273448e-02  2.432823e-02  6.528074e-02  4.712160e-02
5          carts         mem     25  2.162033e+08  2.198430e+08  2.224593e+08  2.219550e+08
6          carts      socket     25  9.500000e+00  1.145000e+01  1.343333e+01  1.305333e+01
10      carts-db         mem     25  7.884390e+07  8.148282e+07  8.425316e+07  8.396829e+07
7          carts    workload     25  7.641183e+00  8.097783e+00  8.655417e+00  8.486297e+00
8       carts-db         cpu     25  3.034738e+00  3.629235e+00  4.388721e+00  4.275059e+00
9       carts-db      diskio     25  1.790057e+06  2.033762e+06  2.296373e+06  2.270569e+06
12     catalogue         cpu     25  1.336217e-01  1.729116e-01  1.858281e-01  1.833097e-01
11      carts-db      socket     25  6.000000e+00  6.000000e+00  7.000000e+00  7.000000e+00
13     catalogue  latency-50     25  3.002940e-03  3.045195e-03  3.110457e-03  3.095734e-03
14     catalogue  latency-90     25  4.605291e-03  4.681351e-03  4.798823e-03  4.772321e-03
46  queue-master         cpu     25  3.252633e-01  4.508656e-01  2.044505e+00  1.407310e+00
15     catalogue         mem     25  6.176768e+06  6.219981e+06  6.272137e+06  6.267372e+06
16     catalogue      socket     25  6.000000e+00  6.000000e+00  6.000000e+00  6.000000e+00
17     catalogue    workload     25  3.925483e+00  4.062383e+00  4.323467e+00  4.260883e+00
18  catalogue-db         cpu     25  1.749846e-01  1.869557e-01  1.977208e-01  1.954904e-01
19  catalogue-db      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00,   earliest_ts latest_ts distinct_cmdb_ids
0        None      None                [], Empty DataFrame
Columns: [cmdb_id, trace_name, count, min_val, median_val, max_val, p95]
Index: [],    earliest_ts   latest_ts                                  distinct_cmdb_ids
0   1705675920  1705677360  [catalogue, front-end, orders, payment, queue-...,          cmdb_id         log_name  count  min_val  median_val  max_val     p95
4      catalogue  log.error_count     25        0         0.0        0     0.0
5      catalogue    log.row_count     25       28       163.0      170   167.8
9         orders    log.row_count     25       22       122.0      134   129.6
8         orders  log.error_count     25        0         0.0        0     0.0
7      front-end    log.row_count     25      355      1966.0     2044  2040.0
6      front-end  log.error_count     25        0         0.0        0     0.0
20          user  log.error_count     25        0         0.0        0     0.0
21          user    log.row_count     25      108       593.0      618   613.4
19      shipping    log.row_count     25       11        61.0       67    64.8
18      shipping  log.error_count     25        0         0.0        0     0.0
13       payment    log.row_count     25       17       101.0      107   104.8
12       payment  log.error_count     25        0         0.0        0     0.0
15  queue-master    log.row_count     25       99       549.0      603   585.6
14  queue-master  log.error_count     25        0         0.0        0     0.0
2       carts-db  log.error_count     24        0         0.0        0     0.0
3       carts-db    log.row_count     24        1         1.0        9     2.0
0          carts  log.error_count     23        0         0.0        0     0.0
1          carts    log.row_count     23       25        50.0      225   146.8
17    session-db    log.row_count      4        5         5.0        5     5.0
16    session-db  log.error_count      4        0         0.0        0     0.0,   earliest_ts latest_ts distinct_cmdb_ids
0        None      None                [], Empty DataFrame
Columns: [cmdb_id, message, count, earliest, latest]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure types and helper ts column
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'].astype('int64'), unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using entire series
global_p95_df = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Merge global_p95 back into metric rows for easy per-row comparisons later
df_metric = df_metric.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 2) Define incident window (UTC) and filter metric rows to that window
start_window = pd.to_datetime('2024-01-19 14:49:49', utc=True)
end_window = pd.to_datetime('2024-01-19 15:19:49', utc=True)
mask_window = (df_metric['ts'] >= start_window) & (df_metric['ts'] <= end_window)
df_window = df_metric[mask_window].copy()

# 3) Restrict to allowed service components
allowed_components = ['carts', 'catalogue', 'orders', 'payment', 'user']

# All distinct (cmdb_id, kpi_name) pairs for allowed components (from full series)
pairs = (
    df_metric[df_metric['cmdb_id'].isin(allowed_components)]
    [['cmdb_id', 'kpi_name']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# 4) For each pair compute requested metrics
# Prepare window subset restricted to allowed components
df_window_allowed = df_window[df_window['cmdb_id'].isin(allowed_components)].copy()

# Total rows in the incident window for restricted components
total_rows_in_window = int(df_window_allowed.shape[0])

# Mark anomalies in the window: value >= global_p95 (global_p95 already merged)
df_window_allowed['is_anomaly'] = df_window_allowed['value'] >= df_window_allowed['global_p95']

# Aggregations from window
agg_window = (
    df_window_allowed
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        total_points_in_window=('value', 'count'),
        max_value_in_window=('value', 'max')
    )
)

# Aggregation for anomalies (count and earliest timestamp)
if not df_window_allowed[df_window_allowed['is_anomaly']].empty:
    agg_anom = (
        df_window_allowed[df_window_allowed['is_anomaly']]
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp', 'min')
        )
    )
else:
    # empty frame with expected columns
    agg_anom = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp'])

# Combine all: start from pairs to ensure inclusion of pairs with zero points in window
result = pairs.merge(global_p95_df, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(agg_window, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(agg_anom, on=['cmdb_id','kpi_name'], how='left')

# Fill missing numeric counts with 0, keep NaNs for max_value and earliest_anomaly_timestamp as None
result['total_points_in_window'] = result['total_points_in_window'].fillna(0).astype(int)
result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)

# Convert earliest_anomaly_timestamp to Python int or None
def to_int_or_none(x):
    if pd.isna(x):
        return None
    try:
        return int(x)
    except:
        return None

result['earliest_anomaly_timestamp'] = result['earliest_anomaly_timestamp'].apply(to_int_or_none)

# breach_ratio = (max_value_in_window - global_p95) / global_p95 (None if global_p95 == 0 or max missing)
def compute_breach(row):
    g = row['global_p95']
    m = row['max_value_in_window']
    if pd.isna(g) or g == 0 or pd.isna(m):
        return None
    return (m - g) / g

result['breach_ratio'] = result.apply(compute_breach, axis=1)

# Keep columns in requested order and make compact
result_compact = result[
    ['cmdb_id', 'kpi_name', 'total_points_in_window', 'anomaly_count',
     'earliest_anomaly_timestamp', 'max_value_in_window', 'global_p95', 'breach_ratio']
]

# Order by anomaly_count desc, then total_points_in_window desc, limit to top 20
result_top20 = result_compact.sort_values(['anomaly_count', 'total_points_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# Final outputs (compact): result_top20 and total_rows_in_window
result_top20, total_rows_in_window

```
Out[2]:
```
Summary of findings (plain English):

- Context: Global P95 thresholds were computed from the full metric series. The incident-window filter was 2024-01-19 14:49:49 UTC to 2024-01-19 15:19:49 UTC. Analysis restricted to services: carts, catalogue, orders, payment, user. Total metric rows in that window for these components: 830.

- What “anomaly” means here: a data point with value >= the global P95 for that (cmdb_id, kpi_name) computed from the entire available series. Note a caveat: if global P95 == 0, comparisons lead to equality with zero being counted as an "anomaly" (breach_ratio is undefined in that case).

- Top results (compact highlights from the returned table, ordered by anomaly_count):
  1. catalogue / socket — total_points=25, anomaly_count=25, earliest_anomaly_timestamp=1705675920, max=6.0, global_P95=6.0, breach_ratio=0.0
  2. orders / error — total_points=25, anomaly_count=25, earliest_anomaly_timestamp=1705675920, max=0.0, global_P95=0.0 (breach_ratio undefined)
  3. user / error — total_points=25, anomaly_count=25, earliest_anomaly_timestamp=1705675920, max=0.0, global_P95=0.0 (breach_ratio undefined)
  4. user / socket — total_points=25, anomaly_count=25, earliest_anomaly_timestamp=1705675920, max=18.0, global_P95=18.0, breach_ratio=0.0
  5. payment / socket — total_points=25, anomaly_count=9, earliest_anomaly_timestamp=1705675920, max=6.0, global_P95=6.0, breach_ratio=0.0
  6. carts / error — total_points=5, anomaly_count=5, earliest_anomaly_timestamp=1705677120, max=0.0, global_P95=0.0 (breach_ratio undefined)
  7. carts / cpu — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705676880, max=79.58958, global_P95≈79.34422, breach_ratio≈0.00309
  8. carts / diskio — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705675920, max=116396.6, global_P95≈59249.02, breach_ratio≈0.96453
  9. carts / latency-50 — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705677180, max≈0.02043347, global_P95≈0.01948694, breach_ratio≈0.04857
  10. carts / latency-90 — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705677180, max≈0.06528074, global_P95≈0.04712160, breach_ratio≈0.38537
  11. carts / mem — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705677180, max≈222,459,300, global_P95≈221,955,000, breach_ratio≈0.00227
  12. carts / socket — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705676700, max≈13.43333, global_P95≈13.05333, breach_ratio≈0.02911
  13. carts / workload — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705676460, max≈8.65542, global_P95≈8.48630, breach_ratio≈0.01993
  14. catalogue / cpu — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705676040, max≈0.1858281, global_P95≈0.1833097, breach_ratio≈0.01374
  15. catalogue / latency-50 — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705677060, max≈0.003110457, global_P95≈0.003095734, breach_ratio≈0.00476
  16. catalogue / latency-90 — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705677060, max≈0.004798823, global_P95≈0.004772321, breach_ratio≈0.00555
  17. catalogue / mem — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705676760, max≈6,272,137, global_P95≈6,267,372, breach_ratio≈0.00076
  18. catalogue / workload — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705676460, max≈4.323467, global_P95≈4.260883, breach_ratio≈0.01469
  19. orders / cpu — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705676340, max≈2.660660, global_P95≈1.785310, breach_ratio≈0.49031
  20. orders / latency-50 — total_points=25, anomaly_count=2, earliest_anomaly_timestamp=1705676880, max≈0.03356718, global_P95≈0.03139213, breach_ratio≈0.06929

- Key takeaways and recommended focus:
  - The strongest single metric breach is carts / diskio (breach_ratio ≈ 0.9645): max value in the window is ~116,396 vs a global P95 of ~59,249 — this suggests a large disk I/O spike for carts and is the highest relative breach.
  - orders / cpu also shows a notable breach (breach_ratio ≈ 0.49).
  - Multiple carts and catalogue KPIs (cpu, mem, latency, socket, workload) show some anomalies (mostly small breach_ratio except diskio and latency-90).
  - Several "error" metrics have global P95 = 0; they are counted as anomalies due to equality with zero—interpret those cautiously (they don't indicate an increase).
  - Overall, prioritize investigating carts disk I/O first, then orders CPU and the carts-related latency spikes. Also validate the meaning of zero-valued P95 "error" metrics before treating them as true errors.

If you want, I can next convert the earliest_anomaly_timestamp values into readable UTC datetimes and produce a filtered list of anomaly timestamps per top KPI for a deeper timeline view.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name  total_points_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window    global_p95  breach_ratio
0   catalogue      socket                      25             25                  1705675920         6.000000e+00  6.000000e+00      0.000000
1      orders       error                      25             25                  1705675920         0.000000e+00  0.000000e+00           NaN
2        user       error                      25             25                  1705675920         0.000000e+00  0.000000e+00           NaN
3        user      socket                      25             25                  1705675920         1.800000e+01  1.800000e+01      0.000000
4     payment      socket                      25              9                  1705675920         6.000000e+00  6.000000e+00      0.000000
5       carts       error                       5              5                  1705677120         0.000000e+00  0.000000e+00           NaN
6       carts         cpu                      25              2                  1705676880         7.958958e+01  7.934422e+01      0.003092
7       carts      diskio                      25              2                  1705675920         1.163966e+05  5.924902e+04      0.964533
8       carts  latency-50                      25              2                  1705677180         2.043347e-02  1.948694e-02      0.048573
9       carts  latency-90                      25              2                  1705677180         6.528074e-02  4.712160e-02      0.385368
10      carts         mem                      25              2                  1705677180         2.224593e+08  2.219550e+08      0.002272
11      carts      socket                      25              2                  1705676700         1.343333e+01  1.305333e+01      0.029111
12      carts    workload                      25              2                  1705676460         8.655417e+00  8.486297e+00      0.019929
13  catalogue         cpu                      25              2                  1705676040         1.858281e-01  1.833097e-01      0.013738
14  catalogue  latency-50                      25              2                  1705677060         3.110457e-03  3.095734e-03      0.004756
15  catalogue  latency-90                      25              2                  1705677060         4.798823e-03  4.772321e-03      0.005553
16  catalogue         mem                      25              2                  1705676760         6.272137e+06  6.267372e+06      0.000760
17  catalogue    workload                      25              2                  1705676460         4.323467e+00  4.260883e+00      0.014688
18     orders         cpu                      25              2                  1705676340         2.660660e+00  1.785310e+00      0.490307
19     orders  latency-50                      25              2                  1705676880         3.356718e-02  3.139213e-02      0.069287, 830)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing variables if present, otherwise load files
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure types and ts columns
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'].astype('int64'), unit='s', utc=True)

df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce').astype('Int64')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
df_log['ts'] = pd.to_datetime(df_log['timestamp'].astype('int64'), unit='s', utc=True)

# Ensure global_p95_df exists (compute if needed) - must be computed from full series per rules
try:
    global_p95_df
except NameError:
    global_p95_df = (
        df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

# Incident window (UTC)
start_window = pd.to_datetime('2024-01-19 14:49:49', utc=True)
end_window = pd.to_datetime('2024-01-19 15:19:49', utc=True)

# Candidate KPIs
candidates = [
    ('carts','diskio'),
    ('carts','cpu'),
    ('carts','latency-90'),
    ('orders','cpu')
]

# Prepare containers for results
group_rows = []
kpi_summary_rows = []

for cmdb_id, kpi in candidates:
    # lookup global P95
    p95_row = global_p95_df[
        (global_p95_df['cmdb_id'] == cmdb_id) & (global_p95_df['kpi_name'] == kpi)
    ]
    if not p95_row.empty:
        global_p95 = float(p95_row['global_p95'].iloc[0])
    else:
        global_p95 = np.nan

    # filter metric rows to window and this KPI
    df_k = df_metric[
        (df_metric['cmdb_id'] == cmdb_id) &
        (df_metric['kpi_name'] == kpi) &
        (df_metric['ts'] >= start_window) &
        (df_metric['ts'] <= end_window)
    ].copy().sort_values('timestamp')

    # total points in window (include for summary)
    total_points_in_window = int(df_k.shape[0])

    # identify anomaly points where value >= global_p95 (NaN p95 leads to no anomalies)
    if np.isnan(global_p95):
        df_anom = df_k.iloc[0:0].copy()  # empty
    else:
        # Comparison works even if p95 == 0 (per earlier logic)
        df_anom = df_k[df_k['value'] >= global_p95].copy()

    anomaly_count = int(df_anom.shape[0])
    earliest_anom_ts = int(df_anom['timestamp'].min()) if anomaly_count > 0 else None

    # Group anomaly points into consecutive sequences where adjacent timestamps differ by <= 60s
    if anomaly_count == 0:
        # still include a summary row
        kpi_summary_rows.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi,
            'total_points_in_window': total_points_in_window,
            'total_anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest_anom_ts
        })
        continue

    df_anom = df_anom.sort_values('timestamp').reset_index(drop=True)
    # compute diff in seconds
    df_anom['delta'] = df_anom['timestamp'].diff().fillna(0).astype('int64')
    # start new group when delta > 60
    df_anom['new_group'] = (df_anom['delta'] > 60).astype(int)
    df_anom['group_id'] = df_anom['new_group'].cumsum()

    # aggregate per group
    grp = df_anom.groupby('group_id', as_index=False).agg(
        cmdb_id=('cmdb_id', 'first'),
        kpi_name=('kpi_name', 'first'),
        group_start_timestamp=('timestamp', 'min'),
        group_end_timestamp=('timestamp', 'max'),
        group_length_in_points=('timestamp', 'count'),
        max_value_in_group=('value', 'max')
    )

    # compute breach_ratio per group
    def calc_breach(max_val, p95):
        if pd.isna(p95) or p95 == 0:
            return None
        return (max_val - p95) / p95

    grp['global_p95'] = global_p95
    grp['breach_ratio'] = grp['max_value_in_group'].apply(lambda mv: calc_breach(mv, global_p95))

    # Append group rows to overall list
    for _, r in grp.iterrows():
        group_rows.append({
            'cmdb_id': r['cmdb_id'],
            'kpi_name': r['kpi_name'],
            'group_start_timestamp': int(r['group_start_timestamp']),
            'group_end_timestamp': int(r['group_end_timestamp']),
            'group_length_in_points': int(r['group_length_in_points']),
            'max_value_in_group': float(r['max_value_in_group']),
            'global_p95': float(r['global_p95']) if not pd.isna(r['global_p95']) else None,
            'breach_ratio': None if pd.isna(r['breach_ratio']) else float(r['breach_ratio'])
        })

    # KPI summary row
    kpi_summary_rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi,
        'total_points_in_window': total_points_in_window,
        'total_anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anom_ts
    })

# Build DataFrames for groups and KPI summaries
groups_df = pd.DataFrame(group_rows,
    columns=['cmdb_id','kpi_name','group_start_timestamp','group_end_timestamp',
             'group_length_in_points','max_value_in_group','global_p95','breach_ratio']
)

kpi_summary_df = pd.DataFrame(kpi_summary_rows,
    columns=['cmdb_id','kpi_name','total_points_in_window','total_anomaly_count','earliest_anomaly_timestamp']
)

# Part 2: carts logs in window
df_logs_carts_window = df_log[
    (df_log['cmdb_id'] == 'carts') &
    (df_log['ts'] >= start_window) &
    (df_log['ts'] <= end_window)
].copy().sort_values('timestamp')

# number of log rows in window
num_log_rows = int(df_logs_carts_window.shape[0])

# stats for log.row_count (filter for log_name == 'log.row_count')
rowcount_rows = df_logs_carts_window[df_logs_carts_window['log_name'] == 'log.row_count']
if not rowcount_rows.empty:
    rowcount_min = float(rowcount_rows['value'].min())
    rowcount_median = float(rowcount_rows['value'].median())
    rowcount_max = float(rowcount_rows['value'].max())
else:
    rowcount_min = rowcount_median = rowcount_max = None

# rows where log.error_count > 0 (filter log_name == 'log.error_count' and value > 0)
error_rows = df_logs_carts_window[
    (df_logs_carts_window['log_name'] == 'log.error_count') &
    (df_logs_carts_window['value'] > 0)
][['timestamp','log_name','value']].sort_values('timestamp').head(20).reset_index(drop=True)

# If none, create an explicit empty DataFrame with columns for clarity
if error_rows.empty:
    carts_error_rows = pd.DataFrame(columns=['timestamp','log_name','value'])
else:
    carts_error_rows = error_rows

# Compact summary dataframe for carts logs
carts_log_summary = pd.DataFrame([{
    'num_log_rows_in_window': num_log_rows,
    'rowcount_min': rowcount_min,
    'rowcount_median': rowcount_median,
    'rowcount_max': rowcount_max,
    'num_error_rows_gt0': int(error_rows.shape[0])
}])

# Final compact outputs
groups_df, kpi_summary_df, carts_log_summary, carts_error_rows

```
Out[3]:
```
Summary (plain English, compact):

1) Anomaly groups (adjacent anomaly points grouped when timestamps differ by <= 60s)

- carts / diskio
  - group: start=1705675920, end=1705675980, length=2 points
  - max_value_in_group = 116396.632691
  - global_P95 = 59249.020593
  - breach_ratio ≈ 0.964533

- carts / cpu
  - group A: start=1705676880, end=1705676880, length=1
    - max = 79.589578, global_P95 = 79.344221, breach_ratio ≈ 0.003092
  - group B: start=1705677120, end=1705677120, length=1
    - max = 79.554044, global_P95 = 79.344221, breach_ratio ≈ 0.002644

- carts / latency-90
  - group A: start=1705677180, end=1705677180, length=1
    - max = 0.06528074, global_P95 = 0.04712160, breach_ratio ≈ 0.385368
  - group B: start=1705677360, end=1705677360, length=1
    - max = 0.047305, global_P95 = 0.047122, breach_ratio ≈ 0.003895

- orders / cpu
  - group: start=1705676340, end=1705676400, length=2 points
  - max_value_in_group = 2.660660
  - global_P95 = 1.785310
  - breach_ratio ≈ 0.490307

2) Per-KPI anomaly summary (within incident window)
- carts / diskio: total_points_in_window = 25, total_anomaly_count = 2, earliest_anomaly_timestamp = 1705675920
- carts / cpu: total_points_in_window = 25, total_anomaly_count = 2, earliest_anomaly_timestamp = 1705676880
- carts / latency-90: total_points_in_window = 25, total_anomaly_count = 2, earliest_anomaly_timestamp = 1705677180
- orders / cpu: total_points_in_window = 25, total_anomaly_count = 2, earliest_anomaly_timestamp = 1705676340

3) carts logs in the incident window (2024-01-19 14:49:49 to 15:19:49 UTC)
- Number of carts log rows in window: 46
- log.row_count statistics (within window): min = 25.0, median = 50.0, max = 225.0
- log.error_count > 0 rows: none found (no rows with log.error_count > 0)

Key takeaway (concise):
- The largest relative breach is carts / diskio (almost a 96% exceedance over its global P95) — this is the most significant anomaly.
- orders / cpu shows a notable breach (≈49% above P95).
- carts shows smaller but present CPU and latency-90 spikes (two short groups each).
- No cart log.error_count > 0 entries were found in the window (logs show activity but no logged errors).

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id    kpi_name  group_start_timestamp  group_end_timestamp  group_length_in_points  max_value_in_group    global_p95  breach_ratio
0   carts      diskio             1705675920           1705675980                       2       116396.632691  59249.020593      0.964533
1   carts         cpu             1705676880           1705676880                       1           79.589578     79.344221      0.003092
2   carts         cpu             1705677120           1705677120                       1           79.554044     79.344221      0.002644
3   carts  latency-90             1705677180           1705677180                       1            0.065281      0.047122      0.385368
4   carts  latency-90             1705677360           1705677360                       1            0.047305      0.047122      0.003895
5  orders         cpu             1705676340           1705676400                       2            2.660660      1.785310      0.490307,   cmdb_id    kpi_name  total_points_in_window  total_anomaly_count  earliest_anomaly_timestamp
0   carts      diskio                      25                    2                  1705675920
1   carts         cpu                      25                    2                  1705676880
2   carts  latency-90                      25                    2                  1705677180
3  orders         cpu                      25                    2                  1705676340,    num_log_rows_in_window  rowcount_min  rowcount_median  rowcount_max  num_error_rows_gt0
0                      46          25.0             50.0         225.0                   0, Empty DataFrame
Columns: [timestamp, log_name, value]
Index: [])```
```